# Save regional PM<sub>2.5</sub> mortality in a single file

Due to memory constraints, regional mortality is saved on an annual, regional and mortality outcome basis. This script combines regional and yearly files into one and saves total mortality based on all mortality outcomes.

In [ ]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [ ]:
# Number of samples
n_samples = 200

In [ ]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/region/{n_samples}_samples/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/"

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")
    for ens_num in ensemble_members:
        print(f"Processing ensemble number {ens_num:02d}")

        ds_years = []
        for year in range(years.start, years.stop+1):

            files = f"Regional_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*_{year}.nc"
            file_path = os.path.join(MORT_DIR, files)
            in_files = sorted(glob.glob(file_path))

            das = []
            for file in in_files:
                da = xr.open_dataarray(file)
                das.append(da)
            combined = xr.concat(das, "region")

            ds_years.append(combined)

        ds_mort = xr.concat(
            ds_years,
            dim=xr.DataArray(
                range(years.start, years.stop+1),
                dims="year",
                name="year")
        )

        description = (f"Regional {health_VAR} mortality due to PM2.5 "
                       " - scripts by A.F. Wells (2025)")
        ds_mort.attrs["description"] = description
        ds_mort.attrs["model"] = model
        ds_mort.attrs["scenario"] = scenario
        ds_mort.attrs["ensemble_number"] = ens_num
        del ds_mort.attrs["year"]
        del ds_mort.attrs["region"]

        out_file = f"Regional_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving {out_path}")
        # encoding ensures long region names are intact
        ds_mort.to_netcdf(out_path, engine="h5netcdf", encoding={"region": {"dtype": str}})

print("All processing complete.")

## Calculate the sum of all mortality outcomes

In [ ]:
for ens_num in ensemble_members:
    # Find all files for this ensemble

    in_files = f"Regional_mortality_*_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(SAVE_DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (important in case of slight coordinate mismatches)
    aligned = xr.align(*datasets, join="exact")

    # Sum across the health variables
    summed_da = sum(aligned)

    description = ("Total regional mortality due to PM2.5 "
                       "- scripts by A.F. Wells (2025)")
    summed_da.attrs["description"] = description
    summed_da.attrs["ensemble_number"] = ens_num
    summed_da.attrs["scenario"] = scenario
    summed_da.attrs["model"] = model

    out_file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving summed mortality timeseries to {out_path}")
    # encoding ensures long region names are intact
    summed_da.to_netcdf(out_path, engine="h5netcdf", encoding={"region": {"dtype": str}})

print("All processing complete.")